In [1]:
from google.colab import drive
import os

# Montaggio del Drive (ti chiederà l'autorizzazione)
drive.mount('/content/drive')

# Installazione del tool ufficiale di Kaggle
!pip install -q kagglehub
print("✅ Ambiente configurato correttamente.")

Mounted at /content/drive
✅ Ambiente configurato correttamente.


In [2]:
import kagglehub

print("⬇️ Scaricamento dataset da Kaggle in corso... (potrebbe volerci qualche minuto)")
# Kagglehub gestisce download, decompressione e caching in automatico
raw_data_path = kagglehub.dataset_download("timoboz/clevr-dataset")
print(f"✅ Dataset scaricato nella cache locale: {raw_data_path}")

⬇️ Scaricamento dataset da Kaggle in corso... (potrebbe volerci qualche minuto)


100%|██████████| 17.7G/17.7G [03:02<00:00, 104MB/s]

Extracting files...


✅ Dataset scaricato nella cache locale: /root/.cache/kagglehub/datasets/timoboz/clevr-dataset/versions/2


In [3]:
import json
import shutil
import random
from tqdm import tqdm

# --- 1. CONFIGURAZIONE PERCORSI SU DRIVE ---
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'

# Qui salveremo gli indici (la mappa del tuo dataset)
INDEX_DIR = os.path.join(BASE_DRIVE, 'indexes')

# Qui salveremo i dati fisici (Immagini e Domande)
PROCESSED_DIR = os.path.join(BASE_DRIVE, 'data/processed')
QUESTIONS_DIR = os.path.join(PROCESSED_DIR, 'questions')

# Creazione cartelle (se non esistono già)
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(QUESTIONS_DIR, exist_ok=True)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(PROCESSED_DIR, 'images', split), exist_ok=True)

# --- 2. FUNZIONE DI RICERCA SICURA ---
def find_file(directory, filename):
    """Esplora le cartelle scaricate senza dare per scontato alcun percorso fisso."""
    for root, dirs, files in os.walk(directory):
        if filename in files:
            return os.path.join(root, filename)
    return None

print("✅ Struttura cartelle su Drive pronta.")

✅ Struttura cartelle su Drive pronta.


In [ ]:
print("⚙️ Generazione degli indici in corso...")

# 1. Troviamo i file JSON originali scaricati
raw_train_json = find_file(raw_data_path, 'CLEVR_train_questions.json')
raw_val_json = find_file(raw_data_path, 'CLEVR_val_questions.json')
raw_test_json = find_file(raw_data_path, 'CLEVR_test_questions.json')

# 2. Carichiamo le liste di domande
with open(raw_train_json, 'r') as f:
    train_full = json.load(f)['questions']
with open(raw_val_json, 'r') as f:
    val_full = json.load(f)['questions']
with open(raw_test_json, 'r') as f:
    test_full = json.load(f)['questions']

# 3. SPLIT DETERMINISTICO (15000, 1000, 1000)
random.seed(42) # FONDAMENTALE per la riproducibilità
random.shuffle(train_full)
random.shuffle(val_full)
random.shuffle(test_full)


# Estraiamo i dizionari (che contengono sia image_filename che question e program)
train_subset = train_full[:15000]
val_subset = val_full[:1000]
test_subset = test_full[:1000]

# 4. SALVATAGGIO INDICI E DOMANDE
# Salviamo con il formato standard di CLEVR {'questions': [...]} così il tuo dataset.py non va modificato
splits = {
    'train': train_subset,
    'val': val_subset,
    'test': test_subset
}

for name, data in splits.items():
    # Salviamo l'indice in /indexes
    with open(os.path.join(INDEX_DIR, f'{name}_index.json'), 'w') as f:
        json.dump({'questions': data}, f)

    # Salviamo una copia anche in /data/processed/questions (per comodità)
    with open(os.path.join(QUESTIONS_DIR, f'{name}_questions.json'), 'w') as f:
        json.dump({'questions': data}, f)

print(f"✅ Indici creati! Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_subset)}")

⚙️ Generazione degli indici in corso...
✅ Indici creati! Train: 5000 | Val: 1000 | Test: 1000


In [ ]:
def copy_images_idempotent(split_name):
    # 1. Carichiamo l'indice appena creato
    index_path = os.path.join(INDEX_DIR, f'{split_name}_index.json')
    with open(index_path, 'r') as f:
        subset = json.load(f)['questions']

    target_dir = os.path.join(PROCESSED_DIR, 'images', split_name)
    print(f"\n--- 🚀 Avvio copia per: {split_name.upper()} ---")

    # 2. Ciclo di copia con tqdm
    for item in tqdm(subset, desc=f"Salvataggio {split_name}"):
        img_name = item['image_filename']
        dst = os.path.join(target_dir, img_name)

        # 3. IL CONTROLLO BLINDATO: Se il file c'è già, non fare nulla
        if not os.path.exists(dst):
            src = find_file(raw_data_path, img_name)
            if src:
                shutil.copy(src, dst)
            else:
                print(f"\n⚠️ ERRORE CRITICO: {img_name} non trovato nella cache originale!")

# Eseguiamo la funzione per tutti e tre i gruppi
copy_images_idempotent('train')
copy_images_idempotent('val')
copy_images_idempotent('test')

print("\n✅ TUTTI I DATI SONO STATI TRASFERITI SU DRIVE CON SUCCESSO!")


--- 🚀 Avvio copia per: TRAIN ---


Salvataggio train: 100%|██████████| 5000/5000 [44:54<00:00,  1.86it/s]



--- 🚀 Avvio copia per: VAL ---


Salvataggio val: 100%|██████████| 1000/1000 [01:01<00:00, 16.23it/s]



--- 🚀 Avvio copia per: TEST ---


Salvataggio test: 100%|██████████| 1000/1000 [01:47<00:00,  9.31it/s]


✅ TUTTI I DATI SONO STATI TRASFERITI SU DRIVE CON SUCCESSO!


In [7]:
import json
import os

print("--- 🔍 VERIFICA INTELLIGENTE DELL'ECOSISTEMA ---")

for split_name in ['train', 'val', 'test']:
    # 1. Leggiamo l'indice per sapere quante DOMANDE abbiamo
    index_path = os.path.join(INDEX_DIR, f'{split_name}_index.json')
    with open(index_path, 'r') as f:
        subset = json.load(f)['questions']

    # 2. Calcoliamo quante immagini UNICHE ci servono davvero
    unique_images = set([item['image_filename'] for item in subset])
    expected_unique_count = len(unique_images)

    # 3. Contiamo i file fisici su Drive
    img_dir = os.path.join(PROCESSED_DIR, 'images', split_name)
    img_count = len([f for f in os.listdir(img_dir) if f.endswith('.png')])

    # 4. Confronto corretto: Immagini fisiche vs Immagini Uniche attese
    status = "✅ OK" if img_count == expected_unique_count else "❌ ERRORE"

    print(f"{status} | {split_name.upper()}:")
    print(f"    - Domande nel JSON: {len(subset)}")
    print(f"    - Immagini uniche necessarie: {expected_unique_count}")
    print(f"    - Immagini trovate su Drive: {img_count}\n")

--- 🔍 VERIFICA INTELLIGENTE DELL'ECOSISTEMA ---
❌ ERRORE | TRAIN:
    - Domande nel JSON: 15000
    - Immagini uniche necessarie: 13619
    - Immagini trovate su Drive: 17597

✅ OK | VAL:
    - Domande nel JSON: 1000
    - Immagini uniche necessarie: 979
    - Immagini trovate su Drive: 979

✅ OK | TEST:
    - Domande nel JSON: 1000
    - Immagini uniche necessarie: 980
    - Immagini trovate su Drive: 980



In [6]:
import os
import json
import random
import shutil
from tqdm import tqdm

# 1. Impostazioni (Aumentiamo a 15.000 per distruggere il bias linguistico)
NUM_TRAIN_SAMPLES = 15000
random.seed(42) # RIPRODUCIBILITÀ GARANTITA

# Percorsi
RAW_TRAIN_JSON = find_file(raw_data_path, 'CLEVR_train_questions.json')
TRAIN_INDEX = os.path.join(INDEX_DIR, 'train_index.json')
TRAIN_IMG_DIR = os.path.join(PROCESSED_DIR, 'images', 'train')

# 2. Creazione del nuovo Indice (Sovrascrive il vecchio da 5k)
print("📝 Generazione nuovo indice...")
with open(RAW_TRAIN_JSON, 'r') as f:
    full_train_data = json.load(f)['questions']

# Selezioniamo 15.000 campioni
train_subset = random.sample(full_train_data, NUM_TRAIN_SAMPLES)

with open(TRAIN_INDEX, 'w') as f:
    json.dump({'questions': train_subset}, f)
print(f"✅ Indice train aggiornato a {NUM_TRAIN_SAMPLES} campioni.")

# 3. Mappa in memoria delle immagini di Kaggle (Velocissimo)
print("🔍 Mappatura immagini raw in corso...")
raw_img_map = {}
for root, dirs, files in os.walk(raw_data_path):
    for file in files:
        if file.endswith('.png'):
            raw_img_map[file] = os.path.join(root, file)

# 4. Copia Chirurgica (Ignora quelle già presenti)
missing_images = []
for item in train_subset:
    img_name = item['image_filename']
    dst = os.path.join(TRAIN_IMG_DIR, img_name)
    if not os.path.exists(dst):
        missing_images.append((img_name, dst))

# Elimina i duplicati dalla lista delle mancanti
missing_images = list(set(missing_images))

if missing_images:
    print(f"⚠️ Trovate {len(missing_images)} nuove immagini da copiare su Drive. Avvio...")
    for img_name, dst in tqdm(missing_images):
        shutil.copy(raw_img_map[img_name], dst)
    print("✅ Copia completata!")
else:
    print("✅ Tutte le immagini sono già presenti su Drive!")

📝 Generazione nuovo indice...
✅ Indice train aggiornato a 15000 campioni.
🔍 Mappatura immagini raw in corso...
⚠️ Trovate 12758 nuove immagini da copiare su Drive. Avvio...


100%|██████████| 12758/12758 [05:51<00:00, 36.25it/s]

✅ Copia completata!


In [4]:
import os
import json
import shutil
from tqdm import tqdm
import time

print("--- 🛡️ AVVIO VALIDATORE E RIPARATORE DATASET ---")

# 1. Quali immagini ci servono DAVVERO?
TRAIN_INDEX = os.path.join(INDEX_DIR, 'train_index.json')
TRAIN_IMG_DIR = os.path.join(PROCESSED_DIR, 'images', 'train')

with open(TRAIN_INDEX, 'r') as f:
    train_data = json.load(f)['questions']

# Estraiamo i nomi unici per evitare di contare la stessa immagine due volte
expected_images = set([item['image_filename'] for item in train_data])
print(f"📌 Immagini uniche richieste dall'indice (15.000 domande): {len(expected_images)}")

# 2. Quali mancano fisicamente su Drive?
missing_images = []
for img_name in expected_images:
    img_path = os.path.join(TRAIN_IMG_DIR, img_name)
    # Controlliamo che il file esista E che non sia corrotto (dimensione > 0 byte)
    if not os.path.exists(img_path) or os.path.getsize(img_path) == 0:
        missing_images.append(img_name)

print(f"⚠️ Immagini mancanti o non scritte correttamente su Drive: {len(missing_images)}")

if len(missing_images) > 0:
    # 3. Mappiamo SOLO i file che ci mancano dalla cartella scaricata da Kaggle
    print("🔍 Ricerca delle immagini mancanti nella cache di Kaggle...")
    raw_img_map = {}
    missing_set = set(missing_images) # Set per ricerca super-veloce

    for root, dirs, files in os.walk(raw_data_path):
        for file in files:
            if file in missing_set:
                raw_img_map[file] = os.path.join(root, file)

    # 4. Copia blindata
    print("🚀 Avvio copia robusta (con verifica)...")
    copied = 0
    for img_name in tqdm(missing_images):
        src_path = raw_img_map.get(img_name)
        if src_path:
            dst_path = os.path.join(TRAIN_IMG_DIR, img_name)
            # copy2 copia anche i metadati per sicurezza
            shutil.copy2(src_path, dst_path)

            # Verifica immediata che il file sia atterrato su Drive
            if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
                copied += 1
        else:
            print(f"\n❌ CRITICO: {img_name} non trovata nei file raw di Kaggle!")

    print(f"\n✅ Copiate e verificate con successo {copied} / {len(missing_images)} immagini.")

    print("⏳ Attesa di 15 secondi per forzare la sincronizzazione server di Google Drive...")
    time.sleep(15)
    print("🏁 Sincronizzazione completata. Ecosistema pronto!")
else:
    print("✅ L'ecosistema è perfetto. Tutte le immagini sono presenti e intatte.")

--- 🛡️ AVVIO VALIDATORE E RIPARATORE DATASET ---
📌 Immagini uniche richieste dall'indice (15.000 domande): 13619
⚠️ Immagini mancanti o non scritte correttamente su Drive: 7514
🔍 Ricerca delle immagini mancanti nella cache di Kaggle...
🚀 Avvio copia robusta (con verifica)...


100%|██████████| 7514/7514 [02:36<00:00, 48.13it/s]



✅ Copiate e verificate con successo 7514 / 7514 immagini.
⏳ Attesa di 15 secondi per forzare la sincronizzazione server di Google Drive...
🏁 Sincronizzazione completata. Ecosistema pronto!
